In [ ]:
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.1.17 langchain-community faiss-cpu transformers sentence-transformers pypdf

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

documents = []

# Create 'data' directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Move uploaded files from root to 'data' directory
# Assuming 'uploaded' variable contains the names of the uploaded files in the current context
# (this would typically come from a previous cell like files.upload())
# Since 'uploaded' is a dictionary from files.upload(), its keys are the filenames.
for filename in uploaded.keys():
    if filename.endswith(".pdf"):
        # Check if the file is already in 'data' to prevent error on re-run
        if not os.path.exists(os.path.join("data", filename)):
            os.rename(filename, os.path.join("data", filename))
            print(f"✅ Moved: {filename} to data/{filename}")

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(f"data/{file}")
        documents.extend(loader.load())

print("✅ Documents loaded:", len(documents))

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("✅ Total chunks:", len(chunks))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embeddings ready")

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)

db.save_local("faiss_index")

print("✅ Database created")

In [ ]:
# Cell 7 — Load FAISS and set up retriever
# k=4 means retrieve top 4 most relevant chunks (configurable)

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

TOP_K = 4  # Change this number to get more or fewer results

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
retriever = db.as_retriever(search_kwargs={"k": TOP_K})

print(f"✅ Retriever ready! Will fetch top {TOP_K} chunks per question")

In [ ]:
# Cell 8B — Generate proper answer using HuggingFace LLM (FREE - no API key needed)
from transformers import pipeline
import textwrap

# Load a better free model for question answering
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

def ask(question):
    print(f"\n❓ Question: {question}")
    print("🔍 Searching documents...\n")

    # Get relevant chunks
    docs = retriever.get_relevant_documents(question)

    # Combine top chunks as context
    context = " ".join([doc.page_content for doc in docs[:3]])
    context = context[:2000]  # limit context size

    # Generate answer using QA model
    try:
        result = qa_pipeline(question=question, context=context)
        answer = result['answer']
        score = result['score']

        print(f"🤖 Answer: {answer}")
        print(f"   Confidence: {score:.0%}")
    except:
        answer = "Could not find a specific answer in the documents."
        print(f"🤖 Answer: {answer}")

    # Show sources
    print(f"\n📚 Sources:")
    seen = set()
    for doc in docs:
        src = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        key = f"{src} | Page {page}"
        if key not in seen:
            print(f"   → {key}")
            seen.add(key)
        # Show chunk preview
        preview = doc.page_content[:200].replace('\n', ' ')
        print(f"     Preview: {preview}...")
    print("\n" + "="*60)

print("✅ Q&A Bot with LLM ready!")

In [ ]:
# Cell 8 — LLM-powered ask() function with citations
from transformers import pipeline

print("⏳ Loading QA model... (takes 1 minute)")
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)
print("✅ QA Model loaded!")

def ask(question):
    print(f"\n❓ Question: {question}")
    print("🔍 Searching documents...\n")

    # Retrieve top chunks
    docs = retriever.get_relevant_documents(question)

    # Combine chunks as context for LLM
    context = " ".join([doc.page_content for doc in docs[:3]])
    context = context[:2000]

    # Generate answer using QA model
    try:
        result = qa_pipeline(question=question, context=context)
        answer = result['answer']
        score = result['score']
        print(f"🤖 Answer: {answer}")
        print(f"   Confidence: {score:.0%}")
    except:
        print("🤖 Answer: Could not find a specific answer in the documents.")

    # Show source citations
    print(f"\n📚 Sources:")
    seen = set()
    for doc in docs:
        src = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        key = f"{src} | Page {page}"
        if key not in seen:
            print(f"   → {key}")
            seen.add(key)
        preview = doc.page_content[:200].replace('\n', ' ')
        print(f"     Chunk: {preview}...")
    print("\n" + "="*60)

In [ ]:
# Cell 9 — Live Q&A Loop
while True:
    question = input("❓ Ask a question (type 'quit' to stop): ").strip()

    if question.lower() == 'quit':
        print("👋 Goodbye!")
        break

    if not question:
        continue

    ask(question)